# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides an end-to-end walkthrough for loading and exploring the FAIR^2 dataset (ordered logistic regression survey results in Northern Kenya) using the `mlcroissant` library.

### Dataset Source
The dataset metadata and schema are hosted at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running locally)
!pip install mlcroissant --quiet

## 1. Data Loading
Load Croissant metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's review the available record sets (`@id`), as well as their field and column identifiers. All Croissant entities are referenced strictly by their `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets by their @id
record_sets = dataset.record_sets
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}")

# For each record set, print its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (@id):")
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"    - {field['@id']}")
        elif isinstance(field, str):
            print(f"    - {field}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print("  Columns (@id):")
    for column in columns:
        if isinstance(column, dict) and '@id' in column:
            print(f"    - {column['@id']}")
        elif isinstance(column, str):
            print(f"    - {column}")

## 3. Data Extraction
Let's load data from each record set into a pandas DataFrame for analysis.
We will use the `@id` of each record set as shown in the previous overview.

In [ ]:
# Retrieve all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns for record set: {record_set_id}")
    except Exception as e:
        print(f"[WARN] Could not load record set {record_set_id}: {e}")

# For demonstration, print the columns of the first record set (if any loaded)
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nSample columns for record set {first_rs}:\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (`@id`) for analysis, filter on a threshold, normalize its distribution, and optionally group the data. Be sure to reference the field/column by its `@id`, as listed previously.

In [ ]:
# Example: Select a numeric field from the first available record set
import numpy as np

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Attempt to pick a likely numeric field
    candidates = [col for col in df.columns if any(word in col.lower() for word in ['loglikelihood', 'iteration', 'value', 'score', 'coef', 'error', 'p_', 'numeric','num','std'])]
    if not candidates:
        candidates = df.select_dtypes(include=np.number).columns.tolist()
    if candidates:
        numeric_field_id = candidates[0]
        print(f"Chosen numeric field: {numeric_field_id}")
        # Use threshold (example: 10)
        threshold = 10
        # Only filter if the numeric field is indeed numeric
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        except Exception:
            pass
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized column '{numeric_field_id}' added:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a categorical/nominal field
        group_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == 'object' or df[col].dtype.name == 'category')]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields detected in the first loaded record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize distributions of a selected numeric field and its grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped data is available, barplot mean by group
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette='mako')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or data available for visualization.")

## 6. Conclusion
In this notebook, we:
- Loaded FAIR^2 regression results and survey metadata via the Croissant schema.
- Programmatically explored record set, field, and column `@id`s with `mlcroissant`.
- Extracted records, selected and normalized a numeric field, filtered and grouped data, and performed basic visualizations.
This approach demonstrates flexible FAIR data access and basic analysis using global interoperable standards.

For further analysis, you can extend EDA to include statistical modeling, missing value imputation, and cross-record set joins using the Croissant `@id` referencing system.